<a href="https://colab.research.google.com/github/charlottesscott/Breast_Screening_Uptake_Prediction/blob/main/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

In [3]:
from google.colab import files

uploaded = files.upload()

Saving master_kc62.csv to master_kc62.csv


In [4]:
Master = pd.read_csv("master_kc62.csv")

In [5]:
df = Master

**Outcome Variable**

Machine learning needs something to predict.

This is called the:

*   Outcome variable
*   Target variable
*   Label
*   Dependent variable

Check available uptake fields

In [6]:
df.columns.tolist()

['collectionyearrange',
 'parent_org_code',
 'parent_org_name',
 'org_name',
 'org_type',
 'age_band',
 'table_code',
 'screened',
 'invited',
 'uptake',
 'total_with_cancer',
 'invasive_total',
 'non_or_micro_invasive',
 'cancer_non_microinvasive',
 'cancer_microinvasive',
 'small_invasive',
 'invasive_not_known',
 'invasive_lessthan10mm',
 'invasive_10mmto15mm',
 'invasive_15mmto20mm',
 'invasive_20mmto50mm',
 'invasive_50mmplus',
 'invasive_unknown',
 'rate_with_cancer',
 'rate_non_or_micro_invasive',
 'rate_small_invasive_x',
 'percent_small_invasive',
 'rate_benign_biopsy',
 'rate_benign_biopsy_warning',
 'nonop_diag_rate_invasive',
 'nonop_diag_rate_noninvasive',
 'rate_non_op_diagnosis',
 'rate_non_op_diagnosis_warning',
 'rate_small_invasive_y',
 'rate_small_invasive_warning',
 'sdr',
 'sdr_warning',
 'initial_referred',
 'percent_assessment',
 'referral_cyt_bio',
 'open_biop_total',
 'final_str',
 'percent_str',
 'org_onscode']

Explore uptake distribution

In [7]:
df['uptake'] = pd.to_numeric(df['uptake'], errors='coerce')
print("Converted 'uptake' column to numeric, coercing errors.")

Converted 'uptake' column to numeric, coercing errors.


In [8]:
df["uptake"].describe()

,uptake
count,8224.000000
mean,65.578243
std,24.330921
min,0.000000
25%,51.382026
50%,70.318105
75%,84.184528
max,100.000000


In [9]:
average_uptake = df['uptake'].mean()
lowest_uptake = df['uptake'].min()
highest_uptake = df['uptake'].max()

quartiles = df['uptake'].quantile([0.25, 0.5, 0.75])

print(f"Average Uptake: {average_uptake:.2f}")
print(f"Lowest Uptake: {lowest_uptake:.2f}")
print(f"Highest Uptake: {highest_uptake:.2f}")
print("\nUptake Quartiles:")
print(quartiles)

Average Uptake: 65.58
Lowest Uptake: 0.00
Highest Uptake: 100.00

Uptake Quartiles:
0.25    51.382026
0.50    70.318105
0.75    84.184528
Name: uptake, dtype: float64


**What counts as low**

In [10]:
threshold = df["uptake"].quantile(0.25)

threshold

np.float64(51.3820262137096)

**Create the target variable**

Low_uptake = 1
High_uptake = 0


CHECK WITH WILL

In [11]:
threshold = df["uptake"].quantile(0.25)

df["low_uptake"] = (
    df["uptake"] < threshold
).astype(int)

In [12]:
df["low_uptake"].value_counts()

,count
low_uptake,
0,6537
1,2056


In [13]:
df[
    ["org_name", "uptake", "low_uptake"]
].head(20)

,org_name,uptake,low_uptake
0,England,66.654372,0
1,England,67.485558,0
2,England,66.637894,0
3,England,66.651002,0
4,England,32.043570,1
5,England,81.499541,0
6,England,64.285714,0
7,England,100.000000,0
8,England,64.775276,0
9,England,81.214704,0


**Feature Engineering**

In [14]:
df.columns.tolist()

['collectionyearrange',
 'parent_org_code',
 'parent_org_name',
 'org_name',
 'org_type',
 'age_band',
 'table_code',
 'screened',
 'invited',
 'uptake',
 'total_with_cancer',
 'invasive_total',
 'non_or_micro_invasive',
 'cancer_non_microinvasive',
 'cancer_microinvasive',
 'small_invasive',
 'invasive_not_known',
 'invasive_lessthan10mm',
 'invasive_10mmto15mm',
 'invasive_15mmto20mm',
 'invasive_20mmto50mm',
 'invasive_50mmplus',
 'invasive_unknown',
 'rate_with_cancer',
 'rate_non_or_micro_invasive',
 'rate_small_invasive_x',
 'percent_small_invasive',
 'rate_benign_biopsy',
 'rate_benign_biopsy_warning',
 'nonop_diag_rate_invasive',
 'nonop_diag_rate_noninvasive',
 'rate_non_op_diagnosis',
 'rate_non_op_diagnosis_warning',
 'rate_small_invasive_y',
 'rate_small_invasive_warning',
 'sdr',
 'sdr_warning',
 'initial_referred',
 'percent_assessment',
 'referral_cyt_bio',
 'open_biop_total',
 'final_str',
 'percent_str',
 'org_onscode',
 'low_uptake']

Convert years into numeric values

Machine learning prefers numbers

In [15]:
df["start_year"] = (
    df["collectionyearrange"]
    .str[:4]
    .astype(int)
)

Create Organisation size feature

WILL TO EXPLAIN

df["size_band"] = pd.qcut(
df["eligible_population"],
q=4,
labels=[
"Small",
"Medium",
"Large",
"Very Large"
]
)

In [16]:
df['screened'] = pd.to_numeric(df['screened'], errors='coerce')
df["screened_size_band"] = pd.qcut(
    df["screened"].dropna(), # Drop NaNs before qcut to avoid errors with missing values
    q=4,
    labels=[
        "Small",
        "Medium",
        "Large",
        "Very Large"
    ],
    duplicates='drop' # Handle cases with identical quantiles
)

In [17]:
df['invited'] = pd.to_numeric(df['screened'], errors='coerce')
df["invited_size_band"] = pd.qcut(
    df["screened"].dropna(), # Drop NaNs before qcut to avoid errors with missing values
    q=4,
    labels=[
        "Small",
        "Medium",
        "Large",
        "Very Large"
    ],
    duplicates='drop' # Handle cases with identical quantiles
)

Historical uptake feature

In [18]:
df = df.sort_values(
    ["org_name", "start_year"]
)

In [21]:
df["previous_uptake"] = (
    df.groupby("org_name")
    ["uptake"]
    .shift(1)
)

Uptake trend

In [22]:
df["uptake_change"] = (
    df["uptake"]
    - df["previous_uptake"]
)

Age Band Encoding

In [23]:
df = pd.get_dummies(
    df,
    columns=["age_band"],
    drop_first=True
)

Region Encoding

In [25]:
df = pd.get_dummies(
    df,
    columns=["parent_org_name"],
    drop_first=True
)

Missing data check - Ask will to explain

In [26]:
df.isnull().sum()

,0
collectionyearrange,0
parent_org_code,0
org_name,0
org_type,0
table_code,0
...,...
parent_org_name_Midlands,0
parent_org_name_North East and Yorkshire,0
parent_org_name_North West,0
parent_org_name_South East,0


In [27]:
df.fillna(
    df.median(numeric_only=True),
    inplace=True
)

Final Feature List

Will to explain

In [28]:
features = [
    "start_year",
    "previous_uptake",
    "uptake_change",
    "eligible_population"
]

In [29]:
X = df.drop(
    columns=[
        "low_uptake",
        "uptake",
        "org_name"
    ]
)

y = df["low_uptake"]